# 03 Forecast Baseline

Explore a first transparent weighted forecast formula. The notebook can tune intuition, but production forecast behavior belongs under `services/api/app/forecasting`.

In [ ]:
from pathlib import Path
import json

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sample = json.loads((REPO_ROOT / "data" / "samples" / "sample_analysis_export.json").read_text())
df = pd.DataFrame(sample["analyses"])
df

In [ ]:
weights = {
    "aggregate_sentiment_score": 0.45,
    "agreement_score": 0.15,
    "evidence_strength_score": 0.2,
    "recent_momentum_score": 0.25,
    "volatility_score": -0.2,
}

def weighted_score(row: pd.Series) -> float:
    return sum(row[column] * weight for column, weight in weights.items())

scored = df.copy()
scored["forecast_score"] = scored.apply(weighted_score, axis=1)
scored["direction"] = scored["forecast_score"].map(
    lambda value: "up" if value > 0.12 else "down" if value < -0.12 else "uncertain"
)
scored["experimental_percent_change"] = (scored["forecast_score"] * 1.2).round(3)
scored[["ticker", "forecast_score", "direction", "experimental_percent_change", "confidence_score"]]

In [ ]:
scored["absolute_error"] = (
    scored["experimental_percent_change"] - scored["actual_percent_change"]
).abs()
scored["baseline_absolute_error"] = (
    scored["baseline_momentum_percent_change"] - scored["actual_percent_change"]
).abs()
scored[["ticker", "absolute_error", "baseline_absolute_error"]]

In [ ]:
report_path = REPO_ROOT / "data" / "reports" / "forecast_baseline_sample.csv"
report_path.parent.mkdir(parents=True, exist_ok=True)
scored.to_csv(report_path, index=False)
report_path

## Promotion Notes

Record promoted weights in `model_versions.parameters` and make the API return `uncertain` when evidence is thin, conflicting, or stale.